In [1]:
import pandas as pd
import numpy as np
from lstm_encoder_decoder import lstm_seq2seq
import torch

In [2]:
file_path = "mid_datav1.csv"

In [3]:
df = pd.read_csv(file_path)

In [4]:
df.head()

,Unnamed: 0,MSG_TYPE,MMSI,NAME,IMO_NUMBER,CALL_SIGN,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,...,time_diff,cog_diff,new_voyage,voyage_id,geometry,in_channel_north,in_channel_south,location,num_pings,include
0,67,1,205089000,DICKENS,9898553.0,ONKZ,21.908479,-76.971564,2023-03-01 08:25:00,13.4,...,1200.0,2.0,True,13_205089000,POINT (-76.971564 21.908479),False,False,east,5,False
1,68,1,205089000,DICKENS,9898553.0,ONKZ,21.901958,-76.959594,2023-03-01 08:30:00,13.2,...,300.0,4.3,False,13_205089000,POINT (-76.959594 21.901958),False,False,east,5,False
2,69,1,205089000,DICKENS,9898553.0,ONKZ,21.895055,-76.938755,2023-03-01 08:35:00,13.2,...,300.0,6.5,False,13_205089000,POINT (-76.938755 21.895055),False,False,east,5,False
3,70,1,205089000,DICKENS,9898553.0,ONKZ,21.876328,-76.888575,2023-03-01 08:45:00,13.4,...,600.0,2.0,False,13_205089000,POINT (-76.888575 21.876328),False,False,east,5,False
4,71,1,205089000,DICKENS,9898553.0,ONKZ,21.872654,-76.878911,2023-03-01 08:50:00,13.4,...,300.0,0.2,False,13_205089000,POINT (-76.878911 21.872654),False,False,east,5,False


In [5]:
print(df.columns)

Index(['Unnamed: 0', 'MSG_TYPE', 'MMSI', 'NAME', 'IMO_NUMBER', 'CALL_SIGN',
       'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG',
       'NAV_STATUS', 'NAV_SENSOR', 'SHIP_AND_CARGO_TYPE', 'DRAUGHT',
       'DRAUGHT.1', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD',
       'DESTINATION', 'MMSI_COUNTRY_CD', 'RECEIVER', 'BEAM', 'LENGTH',
       'CHANNEL_SIDE', 'time_diff', 'cog_diff', 'new_voyage', 'voyage_id',
       'geometry', 'in_channel_north', 'in_channel_south', 'location',
       'num_pings', 'include'],
      dtype='object')


In [4]:
df = df[['MMSI', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'voyage_id']]

In [7]:
df['TIME'] = pd.to_datetime(df['PERIOD'])
df = df.rename(columns = {'LAT_AVG':'LAT', 'LON_AVG':'LON', 'SPEED_KNOTS':'SPEED', 'COG_DEG':'COG', 'HEADING_DEG':'HEADING'})

print(df.dtypes)

MMSI                  int64
LAT                 float64
LON                 float64
PERIOD               object
SPEED               float64
COG                 float64
HEADING             float64
voyage_id            object
TIME         datetime64[ns]
dtype: object


In [8]:
print(len(df))
df = df.dropna()

42883


In [9]:
print(len(df))

42883


In [10]:
df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME
0,205089000,21.908479,-76.971564,2023-03-01 08:25:00,13.4,121.0,119.0,13_205089000,2023-03-01 08:25:00
1,205089000,21.901958,-76.959594,2023-03-01 08:30:00,13.2,116.7,111.0,13_205089000,2023-03-01 08:30:00
2,205089000,21.895055,-76.938755,2023-03-01 08:35:00,13.2,110.2,108.0,13_205089000,2023-03-01 08:35:00
3,205089000,21.876328,-76.888575,2023-03-01 08:45:00,13.4,112.2,109.0,13_205089000,2023-03-01 08:45:00
4,205089000,21.872654,-76.878911,2023-03-01 08:50:00,13.4,112.0,108.0,13_205089000,2023-03-01 08:50:00


In [11]:
df['dt'] = df['TIME'].diff().dt.total_seconds()
df = df.dropna().reset_index(drop=True)


In [12]:
df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt
0,205089000,21.901958,-76.959594,2023-03-01 08:30:00,13.2,116.7,111.0,13_205089000,2023-03-01 08:30:00,300.0
1,205089000,21.895055,-76.938755,2023-03-01 08:35:00,13.2,110.2,108.0,13_205089000,2023-03-01 08:35:00,300.0
2,205089000,21.876328,-76.888575,2023-03-01 08:45:00,13.4,112.2,109.0,13_205089000,2023-03-01 08:45:00,600.0
3,205089000,21.872654,-76.878911,2023-03-01 08:50:00,13.4,112.0,108.0,13_205089000,2023-03-01 08:50:00,300.0
4,205089000,22.547895,-78.097770,2023-09-27 14:10:00,12.9,121.1,121.0,20_205089000,2023-09-27 14:10:00,18163200.0


In [13]:
df['cog_rad'] = np.deg2rad(df['COG'])
df['cog_sin'] = np.sin(df['cog_rad'])
df['cog_cos'] = np.cos(df['cog_rad'])

df['hdg_rad'] = np.deg2rad(df['HEADING'])
df['hdg_sin'] = np.sin(df['hdg_rad'])
df['hdg_cos'] = np.cos(df['hdg_rad'])


In [14]:
df['dlat'] = df['LAT'].diff()
df['dlon'] = df['LON'].diff()

df = df.dropna().reset_index(drop=True)


In [15]:
df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos,dlat,dlon
0,205089000,21.895055,-76.938755,2023-03-01 08:35:00,13.2,110.2,108.0,13_205089000,2023-03-01 08:35:00,300.0,1.923353,0.938493,-0.345298,1.884956,0.951057,-0.309017,-0.006903,0.020839
1,205089000,21.876328,-76.888575,2023-03-01 08:45:00,13.4,112.2,109.0,13_205089000,2023-03-01 08:45:00,600.0,1.958259,0.925871,-0.377841,1.902409,0.945519,-0.325568,-0.018727,0.050180
2,205089000,21.872654,-76.878911,2023-03-01 08:50:00,13.4,112.0,108.0,13_205089000,2023-03-01 08:50:00,300.0,1.954769,0.927184,-0.374607,1.884956,0.951057,-0.309017,-0.003674,0.009664
3,205089000,22.547895,-78.097770,2023-09-27 14:10:00,12.9,121.1,121.0,20_205089000,2023-09-27 14:10:00,18163200.0,2.113594,0.856267,-0.516533,2.111848,0.857167,-0.515038,0.675241,-1.218859
4,205089000,22.523830,-78.055422,2023-09-27 14:20:00,12.9,121.6,122.0,20_205089000,2023-09-27 14:20:00,600.0,2.122320,0.851727,-0.523986,2.129302,0.848048,-0.529919,-0.024065,0.042348


In [16]:
vessels = df['MMSI'].unique()

splitter = int(len(vessels) * 0.9)

train_voyages = vessels[:splitter]
test_voyages  = vessels[splitter:]

df_train = df[df['MMSI'].isin(train_voyages)].copy()
df_test  = df[df['MMSI'].isin(test_voyages)].copy()


print(len(df_train)/len(df))
print(len(df_test)/len(df))
print(len(df_train))
print(len(df_test))



0.8372006249854248
0.16279937501457523
35900
6981


In [17]:
df_train.to_csv("train_data.csv", index = False)
df_test.to_csv("test_data.csv", index = False)

In [18]:
def make_windows_from_df(
    df: pd.DataFrame,
    feature_cols,
    target_cols,
    seq_len: int = 20,
    target_len: int = 5,
    mmsi_col: str = "MMSI",
    time_col: str = "TIME"
):
    """
    Returns:
      X: torch.FloatTensor  (seq_len,  N, n_features)
      Y: torch.FloatTensor  (target_len, N, n_targets)
    """
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])

    X_list = []
    Y_list = []

    for mmsi, g in df.groupby(mmsi_col):
        g = g.sort_values(time_col).reset_index(drop=True)

        feats = g[feature_cols].to_numpy(dtype=np.float32)
        targs = g[target_cols].to_numpy(dtype=np.float32)

        n = len(g)
        if n < seq_len + target_len:
            continue

        for i in range(n - (seq_len + target_len) + 1):
            X_list.append(feats[i : i + seq_len])
            Y_list.append(targs[i + seq_len : i + seq_len + target_len])

    if len(X_list) == 0:
        raise ValueError("No windows created. Increase data or reduce seq_len/target_len.")

    X = torch.from_numpy(np.stack(X_list, axis=0))  # (N, seq_len, n_features)
    Y = torch.from_numpy(np.stack(Y_list, axis=0))  # (N, target_len, 2)

    # Your model expects (seq_len, batch, features)
    X = X.permute(1, 0, 2).contiguous()  # (seq_len, N, n_features)
    Y = Y.permute(1, 0, 2).contiguous()  # (target_len, N, 2)

    return X, Y

In [19]:
feature_cols = ["LAT","LON","SPEED","dt","cog_sin","cog_cos","hdg_sin","hdg_cos"]
target_cols  = ["LAT","LON"]

seq_len = 20
target_len = 5

X_train, Y_train = make_windows_from_df(df_train, feature_cols, target_cols, seq_len=seq_len, target_len=target_len)
X_test,  Y_test  = make_windows_from_df(df_test,  feature_cols, target_cols, seq_len=seq_len, target_len=target_len)

print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_test: ", X_test.shape,  "Y_test: ", Y_test.shape)

X_train: torch.Size([20, 19129, 8]) Y_train: torch.Size([5, 19129, 2])
X_test:  torch.Size([20, 5016, 8]) Y_test:  torch.Size([5, 5016, 2])


In [20]:
def haversine_m(lat1, lon1, lat2, lon2):
    """
    Vectorized haversine distance in meters.
    Inputs can be numpy arrays (any shape) in degrees.
    """
    R = 6371000.0  # meters
    lat1 = np.deg2rad(lat1); lon1 = np.deg2rad(lon1)
    lat2 = np.deg2rad(lat2); lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c

In [21]:
def predict_all(model, X, target_len):
    """
    X: torch tensor (seq_len, N, n_features)
    Returns preds: numpy array (target_len, N, 2)
    """
    model.eval()
    seq_len, N, _ = X.shape
    preds = np.zeros((target_len, N, 2), dtype=np.float32)

    with torch.no_grad():
        for i in range(N):
            one_input = X[:, i, :]              # (seq_len, n_features)
            pred = model.predict(one_input, target_len=target_len)  # (target_len, 2) numpy
            preds[:, i, :] = pred

    return preds

In [22]:
def rmse_mae_meters(pred_latlon, true_latlon):
    """
    pred_latlon, true_latlon: numpy arrays (target_len, N, 2) in degrees
    Returns dict with overall + per-step metrics.
    """
    pred_lat = pred_latlon[..., 0]
    pred_lon = pred_latlon[..., 1]
    true_lat = true_latlon[..., 0]
    true_lon = true_latlon[..., 1]

    err_m = haversine_m(true_lat, true_lon, pred_lat, pred_lon)  # (target_len, N)

    mae_overall = np.mean(np.abs(err_m))
    mse_overall = np.mean(err_m**2)
    rmse_overall = np.sqrt(np.mean(err_m**2))

    return {
        "mae_m": mae_overall,
        "rmse_m": rmse_overall,
        "mse_m": mse_overall,
        "err_m": err_m
    }

In [ ]:
# 1) Predict test set
pred_test = predict_all(model, X_test, target_len=Y_test.shape[0])  # (T, Ntest, 2)

# 2) True test (numpy)
true_test = Y_test.detach().cpu().numpy()



MAE (m):  47032.12
RMSE (m): 56530.06
MASE:     7.665  (denom=6135.57 m)


In [ ]:
# 3) RMSE/MAE in meters
metrics = rmse_mae_meters(pred_test, true_test)

print(f"MAE (m):  {metrics['mae_m']:.2f}")
print(f"RMSE (m): {metrics['rmse_m']:.2f}")
print(f"MSE:  {metrics['mse_m']:.2f}")

MAE (m):  47032.12
RMSE (m): 56530.06
MSE:  47032.12


In [ ]:
# 3) RMSE/MAE in meters
metrics = rmse_mae_meters(pred_test, true_test)

print(f"MAE (m):  {metrics['mae_m']:.2f} meters")
print(f"RMSE (m): {metrics['rmse_m']:.2f} meters")
print(f"MSE:  {metrics['mse_m']:.2f} meters")

MAE (m):  100507.70 meters
RMSE (m): 110894.68 meters
MSE:  12297629696.00 meters


In [ ]:
target_len=[1, 5, 10, 15, 20]
teacher_forcing_ratio_list = [.2, .3, .4, .5, .6, .7]
learning_rate_list = [.01, .04, .06, .08, .1]

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = X_train.to(device)
Y_train = Y_train.to(device)
X_test  = X_test.to(device)
Y_test  = Y_test.to(device)

model = lstm_seq2seq(input_size=X_train.shape[2], hidden_size=128).to(device)

hist = model.train_model(
    input_tensor=X_train,
    target_tensor=Y_train,
    target_len=target_len,
    batch_size=64,
    val_input_tensor=X_test,
    val_target_tensor=Y_test,
    max_epochs=500,
    teacher_forcing_ratio=0.5,
    learning_rate=0.01,
    patience=10,
    min_delta=1e-4,       
    save_path="best_model.pt"
)

  0%|          | 2/500 [09:13<38:15:17, 276.54s/it, no_improve=1, train=0.4193, val=0.4180]  


KeyboardInterrupt: 